# PAN-PC-11 Corpus Parser & Cleaner
### Phase 1 — Semantic-Based Plagiarism Detection

Parses **both external and intrinsic** detection sub-corpora from the PAN Plagiarism Corpus 2011.  
Produces a single `master.csv` in `data/processed/` where **each row = one plagiarism case**.

#### Output columns
| Column | Description |
|---|---|
| `suspicious_id` | Filename of the suspicious document |
| `source_id` | Filename of the source document (NaN for intrinsic) |
| `suspicious_text` | Extracted plagiarised passage from suspicious doc |
| `source_text` | Corresponding source passage (NaN for intrinsic) |
| `is_plagiarism` | 1 = plagiarism case, 0 = non-plagiarism passage |
| `corpus_type` | `external` or `intrinsic` |
| `plagiarism_type` | `artificial`, `simulated`, `manual`, or `none` |
| `obfuscation` | Obfuscation level from XML (if available) |
| `susp_offset` | Character offset in suspicious doc |
| `susp_length` | Character length of suspicious passage |
| `src_offset` | Character offset in source doc |
| `src_length` | Character length of source passage |

## 0. Install / Import Dependencies

In [1]:
# All standard — no extra installs needed
import os
import re
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Optional

import pandas as pd
from tqdm.notebook import tqdm

print('Libraries loaded ✓')

Libraries loaded ✓


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
from pathlib import Path

# Unzip directly in Colab (faster than unzipping on Drive)
zip_path = '/content/drive/MyDrive/pan-corpus-extracted.zip'
extract_to = '/content/'

print('Unzipping... this may take 2-3 minutes')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_to)
print('Done!')

# Verify
import os
for root, dirs, files in os.walk('/content/pan-corpus-extracted'):
    depth = root.replace('/content/pan-corpus-extracted', '').count(os.sep)
    if depth <= 2:
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')

Mounted at /content/drive
Unzipping... this may take 2-3 minutes
Done!
pan-corpus-extracted/
  pan-plagiarism-corpus-2011/
    external-detection-corpus/
    intrinsic-detection-corpus/


## 1. Configuration

In [4]:
# ── Paths ──────────────────────────────────────────────────────────────
BASE_DIR = Path('/content/pan-corpus-extracted/pan-plagiarism-corpus-2011')
OUTPUT_DIR = Path('data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTERNAL_DIR  = BASE_DIR / 'external-detection-corpus'
INTRINSIC_DIR = BASE_DIR / 'intrinsic-detection-corpus'

# ── Cleaner settings ───────────────────────────────────────────────────
MIN_CHARS          = 50
MAX_CHARS          = 10_000
INCLUDE_NON_PLAGIA = True
NON_PLAGIA_CHARS   = 500

# ── Helper: collect files from all part* sub-folders ───────────────────
def glob_parts(parent: Path, pattern: str):
    """Glob a pattern across all part* subdirectories under parent."""
    files = []
    part_dirs = sorted(parent.glob('part*'))
    if part_dirs:
        for part in part_dirs:
            files.extend(sorted(part.glob(pattern)))
    else:
        files.extend(sorted(parent.glob(pattern)))   # flat fallback
    return files

print(f'External corpus : {EXTERNAL_DIR}')
print(f'Intrinsic corpus: {INTRINSIC_DIR}')

# Quick sanity check
susp_xmls = glob_parts(EXTERNAL_DIR / 'suspicious-document', '*.xml')
src_txts  = glob_parts(EXTERNAL_DIR / 'source-document',     '*.txt')
print(f'External suspicious XMLs found : {len(susp_xmls)}')
print(f'External source TXTs found     : {len(src_txts)}')

External corpus : /content/pan-corpus-extracted/pan-plagiarism-corpus-2011/external-detection-corpus
Intrinsic corpus: /content/pan-corpus-extracted/pan-plagiarism-corpus-2011/intrinsic-detection-corpus


External suspicious XMLs found : 11093
External source TXTs found     : 11093


## 2. Helper Utilities

In [5]:
# ── Text helpers ───────────────────────────────────────────────────────

def read_txt(path: Path) -> str:
    """Read a .txt document, trying UTF-8 then latin-1 fallback."""
    try:
        return path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        return path.read_text(encoding='latin-1')


def clean_text(text: str) -> str:
    """
    Lightweight normalisation:
    - Collapse excessive whitespace / newlines
    - Strip leading/trailing whitespace
    - Remove null bytes
    """
    text = text.replace('\x00', '')          # null bytes
    text = re.sub(r'\r\n|\r', '\n', text)   # normalise line endings
    text = re.sub(r'[ \t]+', ' ', text)      # collapse horizontal whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)  # max 2 consecutive blank lines
    return text.strip()


def extract_passage(text: str, offset: int, length: int) -> str:
    """Slice a passage from a document by character offset + length."""
    passage = text[offset: offset + length]
    return clean_text(passage)


def truncate(text: str, max_chars: int = MAX_CHARS) -> str:
    if len(text) > max_chars:
        return text[:max_chars] + ' [TRUNCATED]'
    return text


print('Text helpers defined ✓')

Text helpers defined ✓


In [7]:
# ── XML helpers ────────────────────────────────────────────────────────

def parse_annotation_xml(xml_path: Path) -> list[dict]:
    """
    Parse a PAN-PC-11 annotation XML file.
    Returns a list of feature dicts, one per <feature> element.

    PAN XML structure:
      <document>
        <feature name="plagiarism"
                 this_offset=".." this_length=".."
                 source_reference=".."  (external only)
                 source_offset=".."     (external only)
                 source_length=".."     (external only)
                 type=".."              artificial | simulated | manual
                 obfuscation=".."       />
      </document>
    """
    features = []
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        for feat in root.findall('feature'):
            if feat.get('name') != 'plagiarism':
                continue
            features.append({
                'susp_offset'    : int(feat.get('this_offset', 0)),
                'susp_length'    : int(feat.get('this_length', 0)),
                'source_ref'     : feat.get('source_reference', None),
                'src_offset'     : int(feat.get('source_offset', 0)) if feat.get('source_offset') else None,
                'src_length'     : int(feat.get('source_length', 0)) if feat.get('source_length') else None,
                'plagiarism_type': feat.get('type', 'unknown'),
                'obfuscation'    : feat.get('obfuscation', None),
            })
    except ET.ParseError as e:
        print(f'  [WARN] Could not parse XML {xml_path.name}: {e}')
    return features


print('XML helpers defined ✓')

XML helpers defined ✓


## 3. External Detection Corpus Parser

In [8]:
def parse_external_corpus(external_dir: Path) -> pd.DataFrame:
    susp_dir = external_dir / 'suspicious-document'
    src_dir  = external_dir / 'source-document'

    if not susp_dir.exists():
        print(f'[SKIP] {susp_dir} not found — check path.')
        return pd.DataFrame()

    # Build source-doc lookup: filename → Path (across all parts)
    print('  Indexing source documents…')
    src_index: dict[str, Path] = {}
    for p in glob_parts(src_dir, '*.txt'):
        src_index[p.name] = p

    src_cache: dict[str, str] = {}   # filename → text (lazy loaded)

    xml_files = glob_parts(susp_dir, '*.xml')
    print(f'  Found {len(xml_files)} suspicious-document XML annotations.')

    rows = []
    for xml_path in tqdm(xml_files, desc='External'):
        susp_txt_path = xml_path.with_suffix('.txt')
        if not susp_txt_path.exists():
            continue

        susp_text = read_txt(susp_txt_path)
        features  = parse_annotation_xml(xml_path)
        susp_id   = susp_txt_path.name

        for feat in features:
            susp_passage = extract_passage(susp_text, feat['susp_offset'], feat['susp_length'])
            if len(susp_passage) < MIN_CHARS:
                continue

            src_passage = None
            src_id      = feat['source_ref']

            if src_id:
                # source_ref is just the filename e.g. "source-document00001.txt"
                src_filename = Path(src_id).name
                if src_filename not in src_cache:
                    src_path = src_index.get(src_filename)
                    src_cache[src_filename] = read_txt(src_path) if src_path else ''
                src_text = src_cache[src_filename]
                if src_text and feat['src_offset'] is not None:
                    src_passage = extract_passage(src_text, feat['src_offset'], feat['src_length'])

            rows.append({
                'suspicious_id'   : susp_id,
                'source_id'       : Path(src_id).name if src_id else None,
                'suspicious_text' : truncate(susp_passage),
                'source_text'     : truncate(src_passage) if src_passage else None,
                'is_plagiarism'   : 1,
                'corpus_type'     : 'external',
                'plagiarism_type' : feat['plagiarism_type'],
                'obfuscation'     : feat['obfuscation'],
                'susp_offset'     : feat['susp_offset'],
                'susp_length'     : feat['susp_length'],
                'src_offset'      : feat['src_offset'],
                'src_length'      : feat['src_length'],
            })

        # Negative case
        if INCLUDE_NON_PLAGIA and len(susp_text) > NON_PLAGIA_CHARS:
            covered = set()
            for feat in features:
                covered.update(range(feat['susp_offset'], feat['susp_offset'] + feat['susp_length']))
            neg_passage = None
            for start in range(0, len(susp_text) - NON_PLAGIA_CHARS, 100):
                if not set(range(start, start + NON_PLAGIA_CHARS)) & covered:
                    neg_passage = clean_text(susp_text[start: start + NON_PLAGIA_CHARS])
                    break
            if neg_passage and len(neg_passage) >= MIN_CHARS:
                rows.append({
                    'suspicious_id': susp_id, 'source_id': None,
                    'suspicious_text': neg_passage, 'source_text': None,
                    'is_plagiarism': 0, 'corpus_type': 'external',
                    'plagiarism_type': 'none', 'obfuscation': None,
                    'susp_offset': None, 'susp_length': None,
                    'src_offset': None, 'src_length': None,
                })

    df = pd.DataFrame(rows)
    print(f'  External → {len(df):,} rows ({df["is_plagiarism"].sum():,} positive)')
    return df

print('External parser defined ✓')

External parser defined ✓


## 4. Intrinsic Detection Corpus Parser

In [9]:
def parse_intrinsic_corpus(intrinsic_dir: Path) -> pd.DataFrame:
    susp_dir = intrinsic_dir / 'suspicious-document'

    if not susp_dir.exists():
        print(f'[SKIP] {susp_dir} not found — check path.')
        return pd.DataFrame()

    xml_files = glob_parts(susp_dir, '*.xml')
    print(f'  Found {len(xml_files)} intrinsic suspicious-document XML annotations.')

    rows = []
    for xml_path in tqdm(xml_files, desc='Intrinsic'):
        susp_txt_path = xml_path.with_suffix('.txt')
        if not susp_txt_path.exists():
            continue

        susp_text = read_txt(susp_txt_path)
        features  = parse_annotation_xml(xml_path)
        susp_id   = susp_txt_path.name

        for feat in features:
            susp_passage = extract_passage(susp_text, feat['susp_offset'], feat['susp_length'])
            if len(susp_passage) < MIN_CHARS:
                continue
            rows.append({
                'suspicious_id'   : susp_id,
                'source_id'       : None,
                'suspicious_text' : truncate(susp_passage),
                'source_text'     : None,
                'is_plagiarism'   : 1,
                'corpus_type'     : 'intrinsic',
                'plagiarism_type' : feat['plagiarism_type'],
                'obfuscation'     : feat['obfuscation'],
                'susp_offset'     : feat['susp_offset'],
                'susp_length'     : feat['susp_length'],
                'src_offset'      : None,
                'src_length'      : None,
            })

        if INCLUDE_NON_PLAGIA and len(susp_text) > NON_PLAGIA_CHARS:
            covered = set()
            for feat in features:
                covered.update(range(feat['susp_offset'], feat['susp_offset'] + feat['susp_length']))
            neg_passage = None
            for start in range(0, len(susp_text) - NON_PLAGIA_CHARS, 100):
                if not set(range(start, start + NON_PLAGIA_CHARS)) & covered:
                    neg_passage = clean_text(susp_text[start: start + NON_PLAGIA_CHARS])
                    break
            if neg_passage and len(neg_passage) >= MIN_CHARS:
                rows.append({
                    'suspicious_id': susp_id, 'source_id': None,
                    'suspicious_text': neg_passage, 'source_text': None,
                    'is_plagiarism': 0, 'corpus_type': 'intrinsic',
                    'plagiarism_type': 'none', 'obfuscation': None,
                    'susp_offset': None, 'susp_length': None,
                    'src_offset': None, 'src_length': None,
                })

    df = pd.DataFrame(rows)
    print(f'  Intrinsic → {len(df):,} rows ({df["is_plagiarism"].sum():,} positive)')
    return df

print('Intrinsic parser defined ✓')

Intrinsic parser defined ✓


## 5. Run Both Parsers & Merge

In [10]:
print('=' * 60)
print('Parsing EXTERNAL corpus…')
print('=' * 60)
df_external = parse_external_corpus(EXTERNAL_DIR)

print()
print('=' * 60)
print('Parsing INTRINSIC corpus…')
print('=' * 60)
df_intrinsic = parse_intrinsic_corpus(INTRINSIC_DIR)

# Combine
df_all = pd.concat([df_external, df_intrinsic], ignore_index=True)
print()
print(f'Total rows combined: {len(df_all):,}')

Parsing EXTERNAL corpus…
  Indexing source documents…


  Found 11093 suspicious-document XML annotations.


External:   0%|          | 0/11093 [00:00<?, ?it/s]

  External → 60,465 rows (49,620 positive)

Parsing INTRINSIC corpus…
  Found 4753 intrinsic suspicious-document XML annotations.


Intrinsic:   0%|          | 0/4753 [00:00<?, ?it/s]

  Intrinsic → 16,196 rows (11,443 positive)

Total rows combined: 76,661


/tmp/ipykernel_1454/3686966311.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([df_external, df_intrinsic], ignore_index=True)


## 6. Final Cleaning & Deduplication

In [11]:
print('Running final cleaning pass…')
before = len(df_all)

# ── 1. Drop rows with empty suspicious_text ────────────────────────────
df_all = df_all[df_all['suspicious_text'].notna() & (df_all['suspicious_text'].str.len() >= MIN_CHARS)]

# ── 2. Remove exact duplicates on the text pair ────────────────────────
df_all = df_all.drop_duplicates(subset=['suspicious_id', 'susp_offset', 'susp_length', 'corpus_type'])

# ── 3. Reset index & add a unique case_id ─────────────────────────────
df_all = df_all.reset_index(drop=True)
df_all.insert(0, 'case_id', df_all.index)

# ── 4. Cast dtypes ─────────────────────────────────────────────────────
int_cols = ['susp_offset', 'susp_length', 'src_offset', 'src_length']
for col in int_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce').astype('Int64')  # nullable int

after = len(df_all)
print(f'Rows before cleaning : {before:,}')
print(f'Rows after cleaning  : {after:,}  (dropped {before - after:,})')

Running final cleaning pass…
Rows before cleaning : 76,661
Rows after cleaning  : 76,661  (dropped 0)


## 7. Dataset Summary

In [12]:
print('=' * 60)
print('MASTER CSV — SUMMARY')
print('=' * 60)
print(f"Shape            : {df_all.shape}")
print(f"Columns          : {list(df_all.columns)}")
print()
print('── By corpus_type ──')
print(df_all.groupby('corpus_type')['is_plagiarism'].value_counts().rename('count').to_string())
print()
print('── By plagiarism_type ──')
print(df_all['plagiarism_type'].value_counts().to_string())
print()
print('── By obfuscation (external only) ──')
print(df_all[df_all['corpus_type']=='external']['obfuscation'].value_counts().to_string())
print()
print('── Class balance ──')
vc = df_all['is_plagiarism'].value_counts()
print(f"  Positive (plagiarism)    : {vc.get(1, 0):,}")
print(f"  Negative (non-plagiarism): {vc.get(0, 0):,}")
print(f"  Ratio pos/neg            : {vc.get(1,0)/max(vc.get(0,1),1):.2f}")
print()
print('── Suspicious text length stats ──')
print(df_all['suspicious_text'].str.len().describe().to_string())

MASTER CSV — SUMMARY
Shape            : (76661, 13)
Columns          : ['case_id', 'suspicious_id', 'source_id', 'suspicious_text', 'source_text', 'is_plagiarism', 'corpus_type', 'plagiarism_type', 'obfuscation', 'susp_offset', 'susp_length', 'src_offset', 'src_length']

── By corpus_type ──
corpus_type  is_plagiarism
external     1                49620
             0                10845
intrinsic    1                11443
             0                 4753

── By plagiarism_type ──
plagiarism_type
artificial     49801
none           15598
translation     6654
simulated       4608

── By obfuscation (external only) ──
obfuscation
low     19779
high    19115
none      976

── Class balance ──
  Positive (plagiarism)    : 61,063
  Negative (non-plagiarism): 15,598
  Ratio pos/neg            : 3.91

── Suspicious text length stats ──
count    76661.000000
mean      3204.626133
std       3718.507979
min         84.000000
25%        497.000000
50%       1425.000000
75%       4358.000000
m

## 8. Save Master CSV

In [14]:
OUT_PATH = OUTPUT_DIR / 'master.csv'
df_all.to_csv(OUT_PATH, index=False, encoding='utf-8')
size_mb = OUT_PATH.stat().st_size / 1_048_576
print(f'Saved → {OUT_PATH}  ({size_mb:.1f} MB, {len(df_all):,} rows)')

Saved → data/processed/master.csv  (446.4 MB, 76,661 rows)


## 9. Quick Reload Verification

In [15]:
# Verify the saved CSV loads cleanly for downstream models
df_verify = pd.read_csv(OUT_PATH)
print(f'Reloaded shape  : {df_verify.shape}')
print(f'Columns         : {list(df_verify.columns)}')
print(f'Null suspicious_text rows: {df_verify["suspicious_text"].isna().sum()}')
print()
df_verify.head(10)

Reloaded shape  : (76661, 13)
Columns         : ['case_id', 'suspicious_id', 'source_id', 'suspicious_text', 'source_text', 'is_plagiarism', 'corpus_type', 'plagiarism_type', 'obfuscation', 'susp_offset', 'susp_length', 'src_offset', 'src_length']
Null suspicious_text rows: 0



/tmp/ipykernel_1454/3563285946.py:2: DtypeWarning: Columns (2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_verify = pd.read_csv(OUT_PATH)


,case_id,suspicious_id,source_id,suspicious_text,source_text,is_plagiarism,corpus_type,plagiarism_type,obfuscation,susp_offset,susp_length,src_offset,src_length
0,0,suspicious-document00001.txt,NaN,﻿On a certain morning in March Mrs. Carey sat ...,NaN,0,external,none,NaN,NaN,NaN,NaN,NaN
1,1,suspicious-document00002.txt,NaN,"﻿Up betimes, and to the office receiving lette...",NaN,0,external,none,NaN,NaN,NaN,NaN,NaN
2,2,suspicious-document00003.txt,NaN,﻿THE CONFLICT BETWEEN PRIVATE MONOPOLY AND GOO...,NaN,0,external,none,NaN,NaN,NaN,NaN,NaN
3,3,suspicious-document00004.txt,NaN,﻿OVERLAND.\n\nA Novel\n\nBy\n\nJ. W. DE FOREST...,NaN,0,external,none,NaN,NaN,NaN,NaN,NaN
4,4,suspicious-document00005.txt,source-document00178.txt,"For brown he felt a certain amount of\ns, for ...",". He had come suddenly, unexpectedly, returnin...",1,external,artificial,low,19254.0,1557.0,3835.0,1560.0
5,5,suspicious-document00005.txt,NaN,﻿Marlborough clearly seeing that M. de Vendome...,NaN,0,external,none,NaN,NaN,NaN,NaN,NaN
6,6,suspicious-document00006.txt,NaN,﻿MR. ISAACS A TALE OF MODERN INDIA\n\nBY F. MA...,NaN,0,external,none,NaN,NaN,NaN,NaN,NaN
7,7,suspicious-document00007.txt,source-document06022.txt,That is only archaic by battle of Marathon. An...,. That is easily recollected by the battle of ...,1,external,artificial,low,224.0,2807.0,729730.0,2777.0
8,8,suspicious-document00007.txt,source-document06022.txt,"And, very, when i was asked to address you,\ni...",". And, indeed, when I was asked to address you...",1,external,artificial,low,3032.0,3313.0,134294.0,3307.0
9,9,suspicious-document00007.txt,source-document06022.txt,"You will then find,\non suppressing at part of...",".\nYou will find, on looking at any rich piece...",1,external,artificial,low,6515.0,2923.0,1468563.0,2949.0


---
## Done ✅

Your `data/processed/master.csv` is ready. Load it in any model notebook with:

```python
import pandas as pd
df = pd.read_csv('data/processed/master.csv')

# External only
df_ext = df[df['corpus_type'] == 'external']

# Intrinsic only
df_int = df[df['corpus_type'] == 'intrinsic']

# Positive cases only
df_pos = df[df['is_plagiarism'] == 1]
```

In [16]:
import pandas as pd
import re
from pathlib import Path

# ══════════════════════════════════════════════════════════════════
# 1. PREPROCESSING
# ══════════════════════════════════════════════════════════════════

df = df_all.copy()

# ── A. Fill NaNs with explicit markers (never drop — NaNs are meaningful) 
df['source_id']   = df['source_id'].fillna('NONE')
df['source_text'] = df['source_text'].fillna('NONE')
df['obfuscation'] = df['obfuscation'].fillna('NONE')

# ── B. Flag the ~11k external cases where source_text is unexpectedly missing
df['source_missing'] = (
    (df['corpus_type'] == 'external') &
    (df['is_plagiarism'] == 1) &
    (df['source_text'] == 'NONE')
).astype(int)

print(f"External positive cases with missing source_text: {df['source_missing'].sum():,}")

# ── C. Clean suspicious_text and source_text
def deep_clean(text: str) -> str:
    if not isinstance(text, str) or text == 'NONE':
        return text
    text = re.sub(r'http\S+|www\.\S+', '', text)          # remove URLs
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)            # remove non-ASCII
    text = re.sub(r'\s+', ' ', text)                       # collapse whitespace
    text = re.sub(r'[_\-]{3,}', '', text)                  # remove separator lines
    return text.strip()

print('Cleaning text columns...')
df['suspicious_text'] = df['suspicious_text'].apply(deep_clean)
df['source_text']     = df['source_text'].apply(deep_clean)

# ── D. Drop rows where suspicious_text became empty after cleaning
before = len(df)
df = df[df['suspicious_text'].str.len() >= 50]
print(f'Dropped {before - len(df):,} rows with suspicious_text < 50 chars after cleaning')

# ── E. Add text length columns (useful features for models)
df['susp_text_len'] = df['suspicious_text'].str.len()
df['src_text_len']  = df['source_text'].apply(lambda x: len(x) if x != 'NONE' else 0)

# ── F. Reset index and case_id
df = df.reset_index(drop=True)
df['case_id'] = df.index

# ══════════════════════════════════════════════════════════════════
# 2. SAVE TO GOOGLE DRIVE (persistent) + Colab (fast access)
# ══════════════════════════════════════════════════════════════════

# Save to Drive (permanent)
drive_out = Path('/content/drive/MyDrive/pan-processed')
drive_out.mkdir(parents=True, exist_ok=True)
drive_csv = drive_out / 'master.csv'
df.to_csv(drive_csv, index=False, encoding='utf-8')
print(f'\nSaved to Drive → {drive_csv}')
print(f'Size: {drive_csv.stat().st_size / 1_048_576:.1f} MB')

# Also save to Colab /content for fast access during this session
colab_out = Path('/content/data/processed')
colab_out.mkdir(parents=True, exist_ok=True)
colab_csv = colab_out / 'master.csv'
df.to_csv(colab_csv, index=False, encoding='utf-8')
print(f'Saved to Colab → {colab_csv}')

# ══════════════════════════════════════════════════════════════════
# 3. FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════
print()
print('=== FINAL DATASET SUMMARY ===')
print(f'Shape            : {df.shape}')
print(f'Columns          : {list(df.columns)}')
print()
print('── Class balance ──')
vc = df['is_plagiarism'].value_counts()
print(f'  Positive : {vc.get(1,0):,}')
print(f'  Negative : {vc.get(0,0):,}')
print(f'  Ratio    : {vc.get(1,0)/max(vc.get(0,1),1):.2f}')
print()
print('── NaN check after preprocessing ──')
print(df.isnull().sum())
print()
print('── source_missing breakdown ──')
print(df.groupby('plagiarism_type')['source_missing'].sum())

External positive cases with missing source_text: 0
Cleaning text columns...
Dropped 0 rows with suspicious_text < 50 chars after cleaning

Saved to Drive → /content/drive/MyDrive/pan-processed/master.csv
Size: 445.0 MB
Saved to Colab → /content/data/processed/master.csv

=== FINAL DATASET SUMMARY ===
Shape            : (76661, 16)
Columns          : ['case_id', 'suspicious_id', 'source_id', 'suspicious_text', 'source_text', 'is_plagiarism', 'corpus_type', 'plagiarism_type', 'obfuscation', 'susp_offset', 'susp_length', 'src_offset', 'src_length', 'source_missing', 'susp_text_len', 'src_text_len']

── Class balance ──
  Positive : 61,063
  Negative : 15,598
  Ratio    : 3.91

── NaN check after preprocessing ──
case_id                0
suspicious_id          0
source_id              0
suspicious_text        0
source_text            0
is_plagiarism          0
corpus_type            0
plagiarism_type        0
obfuscation            0
susp_offset        15598
susp_length        15598
src_o